In [ ]:
import joblib
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("--- SENTRi-X: (CIC-IDS2017 Enterprise Data) ---")

processed_dir = "../data/processed/"

# 1. Load the locally scaled CIC-IDS2017 data
print("Loading Enterprise Tensors...")
X_cic = joblib.load(os.path.join(processed_dir, "cic_ids2017_X_test.pkl"))
y_cic = joblib.load(os.path.join(processed_dir, "cic_ids2017_y_test.pkl"))

# 2. Split: 20% for "Studying" (Domain Adaptation), 80% for the Final Exam
X_study, X_exam, y_study, y_exam = train_test_split(
    X_cic, y_cic, test_size=0.80, random_state=42, stratify=y_cic
)

print(f"\nData reserved for SENTRi-X to study: {X_study.shape[0]:,} packets")
print(f"Data reserved for the Final Exam: {X_exam.shape[0]:,} packets")

# 3. Domain Adaptation: Transfer Learning
print("\nInitiating Transfer Learning on the Hybrid Engine (Random Forest Core)...")
rf_adapted = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_adapted.fit(X_study, y_study)

# 4. The Final Evaluation
print("Executing Final Inference...")
adapted_preds = rf_adapted.predict(X_exam)

print(f"\n--- SENTRi-X Performance on Enterprise IT Data ---")
print(f"Final Accuracy: {accuracy_score(y_exam, adapted_preds):.4%}")
print("\nDetailed Report:")
print(classification_report(y_exam, adapted_preds, target_names=['Normal', 'Attack']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_exam, adapted_preds))

--- SENTRi-X: Final Exam (CIC-IDS2017 Enterprise Data) ---
Loading Enterprise Tensors...

Data reserved for SENTRi-X to study: 40,000 packets
Data reserved for the Final Exam: 160,000 packets

Initiating Transfer Learning on the Hybrid Engine (Random Forest Core)...
Executing Final Inference...

--- SENTRi-X Performance on Enterprise IT Data ---
Final Accuracy: 98.1431%

Detailed Report:
              precision    recall  f1-score   support

      Normal       0.99      0.98      0.99    128399
      Attack       0.94      0.97      0.95     31601

    accuracy                           0.98    160000
   macro avg       0.96      0.98      0.97    160000
weighted avg       0.98      0.98      0.98    160000


Confusion Matrix:
[[126289   2110]
 [   861  30740]]
